## PRE-PROCESSING TO CALCULATE INTERCATION FINGERPRINTS

Nicotine acetylcholine receptors alpha 3 beta 4

*anhydrous* / 
    asp; 
    chemplp; 
    chemscore; 
    goldscore

*Hydrated* / 
    asp; 
    chemplp; 
    chemscore; 
    goldscore

a3b4_anh_asp

a3b4_anh_chemplp 

a3b4_anh_chemscore 

a3b4_anh_goldscore  

a3b4_hyd_asp 

a3b4_hyd_chemplp 

a3b4_hyd_chemscore 

a3b4_hyd_goldscore

## We need:

Ligands in path_file with .mol2 format

gold_protein.mol2 file to convert in pdb  for no water analises

In [7]:
import re
import os
from openbabel import pybel as pb
import numpy as np

folder = "a4b2_hyd_chemplp_redocking"

In [8]:
path_file = "../../../../../../Pasta compartilhada entre os PCs/Daniel/PhD_project/phDProject/6-Calc_IFPS/project/inputs/"+folder+"/"

In [9]:
molecules = [f for f in os.listdir(path_file) if (f.endswith(".mol2")) and (f.startswith("_") == False) and (f.endswith("gold_protein.mol2")==False)]
molecules

['CHEMBL667.mol2']

In [10]:
for file_name in molecules:
    #print(file_name)
    file = path_file + file_name
    read_file = open(file).readlines()      #  abrir o arquivo
    
    start_file = read_file[:8]                              #   modificar lineas iniciais do arquivo
    start_file[1] = file_name.split('.mol2')[0]+"\n"
    start_file[3] = "SMALL\n"


# obter coordenadas e ligações

    atoms_list = []
    final_atom = 0
    count = 0
    for l in read_file:                                                                          
        if (re.search("ligands_prepared",l)) and (re.match(" \\d{1,5}",l) != None) and (len(re.findall(" LP ",l))==0): # ATOMS
            count += 1
            final_atom = int(re.match(" \\d{1,5}",l)[0])
            line = l.split(" ")
            if len(str(count)) == 2:
                line[1] = str(count)
            else:
                line[1] = " " + str(count)    
            line[-4] = file_name.split('.mol2')[0]
            print(line)
            line.pop(-5)
            l = " ".join(line)
            atoms_list.append(l)

    initial_atom = final_atom - len(atoms_list)
    num1, num2 = initial_atom, final_atom
    print(atoms_list)

    #print(num1,num2)

    digitos="\d{1,5}"
    primer_intervalo = str(num1)[:4] + "["+str(num1)[4] + "-9]"
    segundo_intervalo = str(num1)[:3] + "["+str(int(str(num1)[3])+1) + "-9][0-9]"
    tercer_intervalo = str(num1)[:2] + "["+str(int(str(num1)[2])+1)+"-9]" + "[0-9][0-9]"
    quarto_intervalo = str(num1)[:1]+ str(int(str(num1)[1])+1) + "[0-9][0-9]"

    reg_expre = " " + digitos + " (" + primer_intervalo + "|" + segundo_intervalo + "|" + tercer_intervalo + "|" + quarto_intervalo + ") (" + primer_intervalo + "|" + segundo_intervalo + "|" + tercer_intervalo + "|" + quarto_intervalo + ")  "   

    #print(reg_expre)
    bonds_list = []
    count_bond = 0
    for l in read_file:
        
        if re.search(reg_expre, l):  # BONDS
            
            count_bond += 1
            line = l.split(" ")
            #print(line)
            if len(str(count_bond)) == 2:
                line[1] = str(count_bond)
            else:
                line[1] = " " + str(count_bond)
            if len(str(int(line[3])-initial_atom)) == 2:
                line[3] =  str(int(line[3])-initial_atom)
            else:
                line[3] =  " " + str(int(line[3])-initial_atom    )
            if len(str(int(line[2])-initial_atom)) == 2:
                line[2] =  str(int(line[2])-initial_atom)
            else:
                line[2] =  " " + str(int(line[2])-initial_atom    )    
            l= " ".join(line) 
            bonds_list.append(l)
    #print(bonds_list)
    if "hyd" in path_file:
        bonds_list = bonds_list[:-4] 
            
        

    # reemplazar dados gerais ( n de atomos e ligações)

    start_file_line_2 = start_file[2].split(" ")
    start_file_line_2.pop(5)

    charge = int(np.array([float(re.findall(" \\d{1}\\.\\d{4}",atm)[0]) for atm in atoms_list]).sum())
    #print(start_file_line_2)
    start_file_line_2[9] = str(charge)
    start_file_line_2[-1] = "1\n"

    n_atoms = str(len(atoms_list))
    start_file_line_2[1] = n_atoms

    bonds_list.insert(0,'@<TRIPOS>BOND\n')
    n_bonds = str(len(bonds_list)-1)
    start_file_line_2[2] = n_bonds

    start_file[2] = " ".join(start_file_line_2)
    #print(" ")
    #print(atoms_list)
    #print(atoms_list+bonds_list+["\n"])
    #rint(" ")
    #rint(" ")

    # Save molEntries File and regroup ligands in one file    

    end_files = open(path_file+"_"+path_file.split("/")[-2]+"_ToFP.mol2","a").writelines(start_file+atoms_list+bonds_list+["\n"])   # multi mol file
    Mol_entries = open(path_file+"_"+path_file.split("/")[-2]+"_MolEntries.txt","a").writelines(file_name.split(".mol2")[0]+'\n')   # entries file
    

    # protein trasnformation mol2-> pdb   (using in hydrated analises)
    if "hyd" in path_file:
        mol_complex = next(pb.readfile("mol2",file))
        output = pb.Outputfile("pdb", file.split(".mol2")[0]+".pdb",overwrite=True)
        output.write(mol_complex)
        output.close()

        mol_complex_cleaner_in = open(file.split(".mol2")[0]+".pdb","r").readlines()
        mol_complex_cleaner = [l for l in mol_complex_cleaner_in if (len(re.findall(" LP | lig | HOH ",l)) == 0) and (l.startswith("ATOM") == True)]
        mol_complex_cleaner = mol_complex_cleaner+["TER   "+str(num1+1)+"     "+re.findall(" [A-Z][A-Z][A-Z] A \d{3}",mol_complex_cleaner[-1])[0]+"\n"]
        mol_complex_cleaner = mol_complex_cleaner+[l for l in mol_complex_cleaner_in if (len(re.findall(" HOH ",l)) > 0) and (len(re.findall(" LP ",l)) == 0) and (l.startswith("ATOM") == True)]
        mol_complex_cleaner = mol_complex_cleaner+["TER   "+str(int(re.findall("\d{5}",mol_complex_cleaner[-1])[0])+1)+"      HOH A 777\n", "END\n"]
        mol_complex_cleaner_out = open(file.split(".mol2")[0]+".pdb","w").writelines(mol_complex_cleaner)

# protein trasnformation mol2-> pdb   (using in anhydrous analises)
if "anh" in path_file:
    protein_file = [f for f in os.listdir(path_file) if (f.startswith("_") == False) and (f.endswith("gold_protein.mol2")==True)][0]  
    #print(protein_file)

    mol_complex = next(pb.readfile("mol2",path_file+protein_file))
    output = pb.Outputfile("pdb", path_file+protein_file.split(".mol2")[0]+".pdb",overwrite=True)
    output.write(mol_complex)
    output.close()

    mol_complex_cleaner_in = open(path_file+protein_file.split(".mol2")[0]+".pdb","r").readlines()
    mol_complex_cleaner = [l for l in mol_complex_cleaner_in if l.startswith("ATOM") == True]
    mol_complex_cleaner = mol_complex_cleaner+["TER   "+str(num1+1)+"     "+re.findall(" [A-Z][A-Z][A-Z] A \d{3}",mol_complex_cleaner[-1])[0]+"\n", "END\n"]
    mol_complex_cleaner_out = open(path_file+protein_file.split(".mol2")[0]+".pdb","w").writelines(mol_complex_cleaner)


['', ' 1', 'N', '', '', '', '', '', '217.2728', '152.4786', '173.6429', '', '', 'N.4', '', '', '', '', '', '778', 'CHEMBL667', '', '', '1.0000\n']
['', ' 2', 'C1', '', '', '', '', '217.5988', '152.8167', '175.0833', '', '', 'C.3', '', '', '', '', '', '778', 'CHEMBL667', '', '', '0.0000\n']
['', ' 3', 'C2', '', '', '', '', '219.0880', '152.8497', '175.4180', '', '', 'C.3', '', '', '', '', '', '778', 'CHEMBL667', '', '', '0.0000\n']
['', ' 4', 'O1', '', '', '', '', '219.1468', '153.1862', '176.8013', '', '', 'O.3', '', '', '', '', '', '778', 'CHEMBL667', '', '', '0.0000\n']
['', ' 5', 'C3', '', '', '', '', '220.3032', '153.0001', '177.4975', '', '', 'C.2', '', '', '', '', '', '778', 'CHEMBL667', '', '', '0.0000\n']
['', ' 6', 'C4', '', '', '', '', '220.4571', '153.5563', '178.8543', '', '', 'C.3', '', '', '', '', '', '778', 'CHEMBL667', '', '', '0.0000\n']
['', ' 7', 'O2', '', '', '', '', '221.1278', '152.3415', '176.8995', '', '', 'O.2', '', '', '', '', '', '778', 'CHEMBL667', '', '', '

*** Open Babel Warning  in Translate
  Cannot perform atom type translation: table cannot find requested types.
*** Open Babel Warning  in ReadMolecule
  This Mol2 file is non-standard. Problem with molecule: a4b2_s23_CORRECTED Cannot interpret atom types correctly, instead attempting to interpret atom type: Lp as elements instead.
*** Open Babel Warning  in Translate
  Cannot perform atom type translation: table cannot find requested types.
*** Open Babel Warning  in ReadMolecule
  This Mol2 file is non-standard. Problem with molecule: a4b2_s23_CORRECTED Cannot interpret atom types correctly, instead attempting to interpret atom type: Lp as elements instead.
*** Open Babel Warning  in ReadMolecule
  Failed to kekulize aromatic bonds in MOL2 file (title is a4b2_s23_CORRECTED)

